# fluency2 Evaluation Report: WMT25 ESA + WMT22–24 MQM
**Algebras AI · April 2026**

Evaluation of fluency2 (structured LLM judge, Gemini 3 Flash,
5 subdimensions) against human translation quality scores
on two independent datasets:
- WMT25 General MT Task (ESA protocol, 11 language pairs)
- WMT22–24 General MT Task (MQM protocol, 6 language pairs)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
# Repo root (run Jupyter from repo root, or from notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
WMT25_PATH = ROOT / 'data/wmt25/merged_all_metrics.parquet'
MQM_PATH = ROOT / 'data/wmt_mqm/all_metrics_mqm.parquet'

# Color palette
COLORS = {
    'fluency2':   '#2D4A8A',
    'gemba':      '#8A4A2D',
    'gemba_mqm':  '#8A4A2D',
    'gemba_da':   '#C4763A',
    'metricx':    '#2D8A5A',
    'xcomet':     '#5A2D8A',
    'cometKiwi':  '#8A7A2D',
    'chrf_pp':    '#8A2D5A',
}

METRIC_LABELS = {
    'fluency2':        'fluency2 (ours)',
    'gemba_score':     'GEMBA-ESA-GPT4.1',
    'gemba_mqm':       'GEMBA-MQM',
    'gemba_da':        'GEMBA-DA',
    'metricx_score':   'MetricX-24-Hybrid-XL',
    'xcomet_score':    'XCOMET-XL',
    'cometKiwi_score': 'CometKiwi-XL',
    'chrf_pp':         'chrF++',
}

SYNTACTIC_DISTANCES = {
    'EN→uk_UA': 0.153, 'EN→is_IS': 0.154, 'EN→ru_RU': 0.183,
    'EN→sr_Cyrl_RS': 0.185, 'EN→et_EE': 0.195, 'EN→zh_CN': 0.268,
    'EN→cs_CZ': 0.335, 'EN→bho_IN': 0.337, 'EN→ar_EG': 0.366,
    'EN→mas_KE': 0.431, 'EN→ja_JP': 0.456, 'EN→it_IT': 0.115,
}

def pairwise_accuracy(pred, human, tie_band=0.0):
    """Fraction of pairs where metric agrees with human ranking."""
    pred = np.array(pred)
    human = np.array(human)
    correct = total = 0
    for i in range(len(pred)):
        for j in range(i+1, len(pred)):
            h_diff = abs(human[i] - human[j])
            if h_diff <= tie_band:
                continue
            total += 1
            if (pred[i] > pred[j]) == (human[i] > human[j]):
                correct += 1
    return correct / total if total > 0 else np.nan

print("Setup complete")
print(f"WMT25 path exists: {WMT25_PATH.exists()}")
print(f"MQM path exists: {MQM_PATH.exists()}")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
df25 = pd.read_parquet(WMT25_PATH)
print(f"WMT25 shape: {df25.shape}")
print(f"Columns: {df25.columns.tolist()}")
print(f"\nRows per pair:")
print(df25.groupby('pair').size().sort_values(ascending=False))
print(f"\nMean human score per pair:")
print(df25.groupby('pair')['human_score'].agg(['mean','std','count']).round(2))

In [ ]:
dfmqm = pd.read_parquet(MQM_PATH)
print(f"MQM shape: {dfmqm.shape}")
print(f"Columns: {dfmqm.columns.tolist()}")
print(f"\nRows per year+pair:")
print(dfmqm.groupby(['year','pair']).size())
print(f"\nMean human MQM score per year+pair:")
print(dfmqm.groupby(['year','pair'])['human_mqm_score'].agg(
    ['mean','std','count']).round(2))
print(f"\nMetric coverage:")
for col in ['fluency2','gemba_mqm','gemba_da','chrf_pp']:
    if col in dfmqm.columns:
        n = dfmqm[col].notna().sum()
        print(f"  {col}: {n}/{len(dfmqm)}")

In [ ]:
def compute_correlations(df, metric_cols, human_col, label=''):
    """Compute Spearman, Kendall, PA for each metric."""
    results = []
    for metric in metric_cols:
        if metric not in df.columns:
            continue
        sub = df[df[metric].notna() & df[human_col].notna()]
        if len(sub) < 10:
            continue
        rho, p_rho = stats.spearmanr(sub[metric], sub[human_col])
        tau, p_tau = stats.kendalltau(sub[metric], sub[human_col])
        pa = pairwise_accuracy(sub[metric].values, sub[human_col].values)
        results.append({
            'metric': metric,
            'label': METRIC_LABELS.get(metric, metric),
            'spearman': round(rho, 3),
            'p_spearman': round(p_rho, 4),
            'kendall': round(tau, 3),
            'pa': round(pa, 3),
            'n': len(sub),
            'dataset': label,
        })
    return pd.DataFrame(results).sort_values('spearman', ascending=False)

print("Helper functions defined")

In [ ]:
WMT25_METRICS = ['fluency2', 'gemba_score', 'metricx_score', 
                  'xcomet_score', 'cometKiwi_score', 'chrf_pp']

# Filter to rows with human_score
df25_scored = df25[df25['human_score'].notna()].copy()

table1 = compute_correlations(df25_scored, WMT25_METRICS, 
                               'human_score', 'WMT25 ESA')

print("=== TABLE 1: WMT25 ESA — Correlation with Human Scores ===")
print(f"n = {len(df25_scored)} rows, 11 language pairs")
print()
display_cols = ['label','spearman','p_spearman','kendall','pa','n']
print(table1[display_cols].to_string(index=False))

# Style the table
styled = table1[display_cols].style\
    .highlight_max(subset=['spearman','pa'], color='#dce6f7')\
    .format({'spearman': '{:.3f}', 'kendall': '{:.3f}', 
             'pa': '{:.3f}', 'p_spearman': '{:.4f}'})
display(styled)

In [ ]:
# Compute PA per LP per metric
lp_results = []
for lp, group in df25_scored.groupby('pair'):
    dist = SYNTACTIC_DISTANCES.get(lp, np.nan)
    for metric in WMT25_METRICS:
        if metric not in group.columns:
            continue
        sub = group[group[metric].notna() & group['human_score'].notna()]
        if len(sub) < 5:
            continue
        pa = pairwise_accuracy(sub[metric].values, sub['human_score'].values)
        lp_results.append({
            'pair': lp, 'syntactic_distance': dist,
            'metric': metric, 'pa': round(pa, 3), 'n': len(sub)
        })

df_lp = pd.DataFrame(lp_results)
table3 = df_lp.pivot_table(
    index=['pair','syntactic_distance'], 
    columns='metric', values='pa'
).reset_index().sort_values('syntactic_distance')

print("=== TABLE 3: Pairwise Accuracy by Language Pair ===")
print("Sorted by syntactic distance (lang2vec)")
print()
print(table3.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

metrics_to_plot = ['fluency2', 'gemba_score', 'metricx_score', 
                   'chrf_pp', 'xcomet_score']
pairs_sorted = table3['pair'].tolist()
x = np.arange(len(pairs_sorted))
width = 0.15

for i, metric in enumerate(metrics_to_plot):
    if metric not in table3.columns:
        continue
    vals = [table3[table3['pair']==p][metric].values[0] 
            if len(table3[table3['pair']==p]) > 0 else np.nan 
            for p in pairs_sorted]
    color = COLORS.get(metric.replace('_score',''), '#888888')
    label = METRIC_LABELS.get(metric, metric)
    ax.bar(x + i*width, vals, width, label=label, 
           color=color, alpha=0.85)

ax.set_xlabel('Language Pair (sorted by syntactic distance →)')
ax.set_ylabel('Pairwise Accuracy')
ax.set_title('Pairwise Accuracy by Language Pair — WMT25 ESA')
ax.set_xticks(x + width*2)
ax.set_xticklabels([p.replace('EN→','') for p in pairs_sorted], 
                   rotation=45, ha='right')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, 
           label='Random baseline')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.savefig('/tmp/fig_table3_pa_by_lp.png', dpi=150)
plt.show()
print("fluency2 best on 9/11 pairs")

In [ ]:
from scipy.stats import spearmanr as sp

# Compute judge advantage and typological correlation
typo_results = []
baselines = ['gemba_score', 'metricx_score', 'xcomet_score', 
             'cometKiwi_score', 'chrf_pp']

# Get PA per LP for fluency2
pa_fluency2 = {}
for lp, group in df25_scored.groupby('pair'):
    sub = group[group['fluency2'].notna() & group['human_score'].notna()]
    if len(sub) >= 5:
        pa_fluency2[lp] = pairwise_accuracy(
            sub['fluency2'].values, sub['human_score'].values)

for baseline in baselines:
    if baseline not in df25_scored.columns:
        continue
    
    advantages = []
    distances = []
    lp_names = []
    
    for lp, group in df25_scored.groupby('pair'):
        dist = SYNTACTIC_DISTANCES.get(lp, np.nan)
        if np.isnan(dist) or lp not in pa_fluency2:
            continue
        sub = group[group[baseline].notna() & group['human_score'].notna()]
        if len(sub) < 5:
            continue
        pa_base = pairwise_accuracy(
            sub[baseline].values, sub['human_score'].values)
        adv = pa_fluency2[lp] - pa_base
        advantages.append(adv)
        distances.append(dist)
        lp_names.append(lp)
    
    if len(advantages) >= 5:
        rho, p = sp(distances, advantages)
        typo_results.append({
            'baseline': baseline,
            'label': METRIC_LABELS.get(baseline, baseline),
            'rho': round(rho, 3),
            'p_value': round(p, 4),
            'n_lp': len(advantages),
            'significant': p < 0.05
        })

df_typo = pd.DataFrame(typo_results)
print("=== TYPOLOGICAL ANALYSIS ===")
print("Spearman(syntactic_distance, fluency2_advantage_vs_baseline)")
print()
print(df_typo[['label','rho','p_value','significant','n_lp']].to_string(index=False))

In [ ]:
# Recompute advantage data for chrF++ plot
adv_data = []
for lp, group in df25_scored.groupby('pair'):
    dist = SYNTACTIC_DISTANCES.get(lp, np.nan)
    if np.isnan(dist) or lp not in pa_fluency2:
        continue
    sub = group[group['chrf_pp'].notna() & group['human_score'].notna()]
    if len(sub) < 5:
        continue
    pa_chrf = pairwise_accuracy(sub['chrf_pp'].values, sub['human_score'].values)
    adv_data.append({
        'lp': lp.replace('EN→',''),
        'dist': dist,
        'advantage': pa_fluency2[lp] - pa_chrf
    })

df_adv = pd.DataFrame(adv_data)
rho_val = df_typo[df_typo['baseline']=='chrf_pp']['rho'].values[0]
p_val = df_typo[df_typo['baseline']=='chrf_pp']['p_value'].values[0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_adv['dist'], df_adv['advantage'], 
           color=COLORS['fluency2'], s=80, zorder=5)

# Regression line
m, b = np.polyfit(df_adv['dist'], df_adv['advantage'], 1)
x_line = np.linspace(df_adv['dist'].min(), df_adv['dist'].max(), 100)
ax.plot(x_line, m*x_line + b, color=COLORS['chrf_pp'], 
        linewidth=2, alpha=0.7)

for _, row in df_adv.iterrows():
    ax.annotate(row['lp'], (row['dist'], row['advantage']),
                textcoords='offset points', xytext=(5,5), fontsize=9)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Syntactic Distance (lang2vec)', fontsize=11)
ax.set_ylabel('fluency2 PA − chrF++ PA', fontsize=11)
ax.set_title(f'fluency2 Advantage over chrF++ vs Syntactic Distance\n'
             f'Spearman ρ = {rho_val}, p = {p_val}', fontsize=11)
plt.tight_layout()
plt.savefig('/tmp/fig_typological_scatter.png', dpi=150)
plt.show()

In [ ]:
loo_results = []
lp_list = df_adv['lp'].tolist()
dist_list = df_adv['dist'].tolist()
adv_list = df_adv['advantage'].tolist()

for drop_i, drop_lp in enumerate(lp_list):
    remaining_dist = [d for i,d in enumerate(dist_list) if i != drop_i]
    remaining_adv = [a for i,a in enumerate(adv_list) if i != drop_i]
    rho, _ = sp(remaining_dist, remaining_adv)
    loo_results.append({'dropped': drop_lp, 'rho': round(rho, 3)})

df_loo = pd.DataFrame(loo_results)
print("=== LOO SENSITIVITY (fluency2 vs chrF++) ===")
print(f"Full sample ρ = {rho_val}")
print(df_loo.to_string(index=False))
print(f"\nRange: [{df_loo['rho'].min()}, {df_loo['rho'].max()}]")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_loo['dropped'], df_loo['rho'], 
       color=COLORS['chrf_pp'], alpha=0.8)
ax.axhline(rho_val, color=COLORS['fluency2'], 
           linewidth=2, linestyle='--', label=f'Full sample ρ={rho_val}')
ax.set_xlabel('Dropped Language Pair')
ax.set_ylabel('Spearman ρ')
ax.set_title('LOO Sensitivity: fluency2 vs chrF++')
ax.legend()
ax.set_ylim(-1, 0)
plt.tight_layout()
plt.savefig('/tmp/fig_loo.png', dpi=150)
plt.show()

In [ ]:
MQM_METRICS = [c for c in ['fluency2','gemba_mqm','gemba_da','chrf_pp'] 
               if c in dfmqm.columns]

dfmqm_scored = dfmqm[dfmqm['human_mqm_score'].notna()].copy()

table_mqm = compute_correlations(
    dfmqm_scored, MQM_METRICS, 
    'human_mqm_score', 'WMT22-24 MQM'
)

print("=== WMT22-24 MQM — Correlation with Human MQM Scores ===")
print(f"n = {len(dfmqm_scored)} rows total")
print(f"Note: MQM scale (0=perfect, negative=errors)")
print(f"GEMBA scores available for WMT22 only (en-de, zh-en, en-ru)")
print()
display_cols = ['label','spearman','p_spearman','kendall','pa','n']
print(table_mqm[display_cols].to_string(index=False))
display(table_mqm[display_cols].style\
    .highlight_max(subset=['spearman','pa'], color='#dce6f7')\
    .format({'spearman':'{:.3f}','kendall':'{:.3f}',
             'pa':'{:.3f}','p_spearman':'{:.4f}'}))

In [ ]:
lp_mqm = []
for (year, pair), group in dfmqm_scored.groupby(['year','pair']):
    for metric in MQM_METRICS:
        sub = group[group[metric].notna() & 
                   group['human_mqm_score'].notna()]
        if len(sub) < 5:
            continue
        rho, _ = stats.spearmanr(sub[metric], sub['human_mqm_score'])
        pa = pairwise_accuracy(sub[metric].values, 
                               sub['human_mqm_score'].values)
        lp_mqm.append({
            'year': year, 'pair': pair, 'metric': metric,
            'label': METRIC_LABELS.get(metric, metric),
            'spearman': round(rho,3), 'pa': round(pa,3), 'n': len(sub)
        })

df_lp_mqm = pd.DataFrame(lp_mqm)
print("=== MQM per Year + Language Pair ===")
pivot = df_lp_mqm.pivot_table(
    index=['year','pair'], columns='label', 
    values='pa'
).round(3)
print(pivot.to_string())

In [ ]:
print("=== CROSS-DATASET COMPARISON: ESA vs MQM ===\n")

comparison = []
for metric in ['fluency2', 'chrf_pp']:
    # ESA
    esa_row = table1[table1['metric']==metric]
    # MQM  
    mqm_row = table_mqm[table_mqm['metric']==metric]
    
    if len(esa_row) > 0:
        comparison.append({
            'metric': METRIC_LABELS.get(metric, metric),
            'ESA_spearman': esa_row['spearman'].values[0],
            'ESA_pa': esa_row['pa'].values[0],
            'ESA_n': esa_row['n'].values[0],
            'MQM_spearman': mqm_row['spearman'].values[0] if len(mqm_row) > 0 else None,
            'MQM_pa': mqm_row['pa'].values[0] if len(mqm_row) > 0 else None,
            'MQM_n': mqm_row['n'].values[0] if len(mqm_row) > 0 else None,
        })

df_comp = pd.DataFrame(comparison)
print(df_comp.to_string(index=False))

# Also add GEMBA comparison
print("\n--- GEMBA on MQM (WMT22 only) ---")
for metric in ['gemba_mqm', 'gemba_da']:
    mqm_row = table_mqm[table_mqm['metric']==metric]
    if len(mqm_row) > 0:
        print(f"{METRIC_LABELS.get(metric,metric)}: "
              f"Spearman={mqm_row['spearman'].values[0]}, "
              f"PA={mqm_row['pa'].values[0]}, "
              f"n={mqm_row['n'].values[0]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ESA barplot
ax = axes[0]
esa_data = table1.sort_values('spearman', ascending=True)
colors_esa = [COLORS.get(m.replace('_score',''), '#888') 
              for m in esa_data['metric']]
bars = ax.barh(esa_data['label'], esa_data['spearman'], 
               color=colors_esa, alpha=0.85)
ax.set_xlabel('Spearman ρ')
ax.set_title('WMT25 ESA\n(n=3,274, 11 pairs)')
ax.set_xlim(0, 0.75)
for bar, val in zip(bars, esa_data['spearman']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# MQM barplot
ax = axes[1]
mqm_data = table_mqm.sort_values('spearman', ascending=True)
colors_mqm = [COLORS.get(m.replace('_score',''), '#888') 
              for m in mqm_data['metric']]
bars = ax.barh(mqm_data['label'], mqm_data['spearman'],
               color=colors_mqm, alpha=0.85)
ax.set_xlabel('Spearman ρ')
ax.set_title('WMT22-24 MQM\n(fluency2/chrf: n=2658/824, GEMBA: n=824)')
ax.set_xlim(0, 0.75)
for bar, val in zip(bars, mqm_data['spearman']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('fluency2 vs Baseline Metrics — Spearman Correlation with Human Scores',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_summary_comparison.png', dpi=150)
plt.show()

In [ ]:
subdims = [c for c in ['idiomatic','collocational','discourse',
                        'pragmatic','calque'] 
           if c in df25_scored.columns]

if subdims:
    print("=== SUBDIMENSION ANALYSIS (WMT25 ESA) ===")
    subdim_results = []
    for dim in subdims:
        sub = df25_scored[df25_scored[dim].notna() & 
                         df25_scored['human_score'].notna()]
        if len(sub) < 10:
            continue
        rho, p = stats.spearmanr(sub[dim], sub['human_score'])
        subdim_results.append({
            'subdimension': dim, 
            'spearman': round(rho,3),
            'p': round(p,4),
            'n': len(sub)
        })
    df_subdim = pd.DataFrame(subdim_results).sort_values(
        'spearman', ascending=False)
    print(df_subdim.to_string(index=False))
    
    # Plot
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(df_subdim['subdimension'], df_subdim['spearman'],
           color=COLORS['fluency2'], alpha=0.8)
    ax.set_ylabel('Spearman ρ vs human score')
    ax.set_title('fluency2 Subdimension Correlations (WMT25 ESA)')
    plt.tight_layout()
    plt.savefig('/tmp/fig_subdimensions.png', dpi=150)
    plt.show()
else:
    print("No subdimension columns found in WMT25 data")

# MQM subdimensions
subdims_mqm = [c for c in ['idiomatic','collocational','discourse',
                             'pragmatic','calque'] 
               if c in dfmqm_scored.columns]
if subdims_mqm:
    print("\n=== SUBDIMENSION ANALYSIS (WMT22-24 MQM) ===")
    for dim in subdims_mqm:
        sub = dfmqm_scored[dfmqm_scored[dim].notna() & 
                           dfmqm_scored['human_mqm_score'].notna()]
        if len(sub) < 10:
            continue
        rho, p = stats.spearmanr(sub[dim], sub['human_mqm_score'])
        print(f"  {dim}: ρ={rho:.3f}, p={p:.4f}, n={len(sub)}")

## Limitations

**WMT25 ESA:**
- n = 11 language pairs for typological analysis — wide CIs
- 20.5% rows lost in join (refA + speech domain) — equal for all metrics
- EN→it_IT excluded (no GEMBA scores in automatic_scores)
- GEMBA, MetricX, XCOMET, CometKiwi taken from official WMT25 repo 
  without recomputation

**WMT22-24 MQM:**
- GEMBA scores available for WMT22 only (3 pairs: en-de, zh-en, en-ru)
- chrF++ only for rows with ref_text (824/2658 rows)
- No MetricX/XCOMET scores (mt-metrics-eval-v2 inaccessible)

**Both datasets:**
- fluency2 backbone: Gemini 3 Flash only — not tested with other models
- Potential data contamination: Gemini 3 Flash training cutoff 
  after WMT25 segments published
- EN→mas_KE: fluency2 PA=0.550 (all systems score very low, mean=3.7)
- Typological pattern (ρ=−0.827) significant only vs chrF++, 
  not vs neural metrics (p>0.29)

## Reproducing These Results

**Data preparation**
- WMT25: `tools/merge_wmt25_auto_metrics.py`
- WMT22–24: `tools/wmt_mqm_historical_pipeline.py` + `tools/merge_gemba_into_wmt_mqm.py`

**fluency2 computation**
```bash
python3 algebras-ml/tools/compute_fluency2_gemini.py \
  --input <input.parquet> \
  --output <output.parquet> \
  --max-concurrent 20 --resume
```

**Analysis**
- `tools/wmt25_full_correlation_analysis.py`

In [ ]:
print("All results computed from:")
print(f"  WMT25: {WMT25_PATH}")
print(f"  MQM:   {MQM_PATH}")
print(f"  fluency2 model: Gemini 3 Flash (gemini-3-flash-preview)")